# Chapter 2: Reinforcement Learning Fundamentals
### **RL: The Seminal Papers** by Rahul Shirale

Welcome to the interactive companion notebook for Chapter 2. In this notebook, we will explore the fundamental objects of RL: **Markov Decision Processes (MDPs)**, **Temporal Difference (TD) Learning**, and the classic **Cliff Walking** benchmark comparing Q-Learning and SARSA.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshirale/rl-seminal-papers/blob/main/src/part_1_foundations/ch02_fundamentals/Chapter2_Fundamentals.ipynb)

## 1. Setup
We'll start by importing our dependencies. In Google Colab, these are pre-installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd # For pretty table display
import os

# Set seed for reproducibility
np.random.seed(42)

## 2. The Environments
We define two core environments. Note that we use a standardized **transition interface** where every step returns `(next_state, reward, done)`.

In [ ]:
class GridWorld:
    def __init__(self, width=4, height=3):
        self.width, self.height = width, height
        self.actions = ['Up', 'Down', 'Left', 'Right']
        self.action_map = {'Up': (0, 1), 'Down': (0, -1), 'Right': (1, 0), 'Left': (-1, 0)}
        self.terminals = {(3, 2): 10, (1, 1): -10, (2, 1): -10}

    def transition(self, state, action):
        if state in self.terminals: return state, 0, True
        dx, dy = self.action_map[action]
        next_s = (state[0] + dx, state[1] + dy)
        if not (0 <= next_s[0] < self.width and 0 <= next_s[1] < self.height): next_s = state
        
        reward = self.terminals.get(next_s, -1)
        done = next_s in self.terminals
        return next_s, reward, done

class CliffWalking:
    def __init__(self, width=12, height=4):
        self.width, self.height = width, height
        self.actions = ['Up', 'Down', 'Left', 'Right']
        self.action_map = {'Up': (0, 1), 'Down': (0, -1), 'Right': (1, 0), 'Left': (-1, 0)}
        self.start, self.goal = (0, 0), (11, 0)
        self.cliff = [(x, 0) for x in range(1, 11)]

    def transition(self, state, action):
        if state == self.goal: return state, 0, True
        dx, dy = self.action_map[action]
        next_s = (state[0] + dx, state[1] + dy)
        if not (0 <= next_s[0] < self.width and 0 <= next_s[1] < self.height): next_s = state
        
        if next_s in self.cliff: return self.start, -100, False
        return next_s, -1, (next_s == self.goal)

## 3. The Algorithms
Implementation of TD(0), Q-Learning, and SARSA. By standardizing the environment interface, our algorithm code remains clean and environment-agnostic.

In [ ]:
def td0(env, episodes=1000, alpha=0.1, gamma=0.99):
    V = {(x, y): 0.0 for x in range(env.width) for y in range(env.height)}
    for _ in range(episodes):
        state, done = (0, 0), False
        while not done:
            action = np.random.choice(env.actions)
            next_s, reward, done = env.transition(state, action)
            V[state] += alpha * (reward + gamma * V[next_s] - V[state])
            state = next_s
    return V

def run_agent(env, mode='qlearning', episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = {((x, y), a): 0.0 for x in range(env.width) for y in range(env.height) for a in env.actions}
    
    def choose_action(s):
        if np.random.random() < epsilon: return np.random.choice(env.actions)
        q_vals = [Q[(s, a)] for a in env.actions]; max_q = max(q_vals)
        return np.random.choice([a for a, q in zip(env.actions, q_vals) if q == max_q])

    falls, history = 0, []
    for _ in range(episodes):
        s = getattr(env, 'start', (0, 0))
        a = choose_action(s)
        total_r, done = 0, False
        while not done:
            next_s, r, done = env.transition(s, a)
            total_r += r
            if r == -100: falls += 1
            
            next_a = choose_action(next_s)
            
            if mode == 'qlearning':
                target = r + gamma * max(Q[(next_s, act)] for act in env.actions) * (not done)
            else:
                target = r + gamma * Q[(next_s, next_a)] * (not done)
                
            Q[(s, a)] += alpha * (target - Q[(s, a)])
            s, a = next_s, next_a
        history.append(total_r)
    return Q, history, falls

## 4. Experiment 1: TD(0) Random Policy Results
How often does a random walker survive the Grid World?

In [ ]:
env = GridWorld()
outcomes = {"Goal": 0, "Hazard (1,1)": 0, "Hazard (2,1)": 0}
for _ in range(500):
    s, done = (0, 0), False
    while not done: s, _, done = env.transition(s, np.random.choice(env.actions))
    if s == (3, 2): outcomes["Goal"] += 1
    else: outcomes[f"Hazard {s}"] += 1

print("Random Policy Outcomes (500 episodes):")
for k, v in outcomes.items(): print(f"  - {k}: {v} ({v/5:.1f}%)")

## 5. Experiment 2: Cliff Walking (Brave vs Cautious)
Comparing Q-Learning (Off-policy) and SARSA (On-policy).

In [ ]:
env = CliffWalking()
_, q_hist, q_falls = run_agent(env, mode='qlearning')
_, s_hist, s_falls = run_agent(env, mode='sarsa')

plt.figure(figsize=(10, 5))
plt.plot(pd.Series(q_hist).rolling(20).mean(), label="Q-Learning (Brave)", color="#D97706")
plt.plot(pd.Series(s_hist).rolling(20).mean(), label="SARSA (Cautious)", color="#3498db")
plt.title("Reward per Episode (Moving Average)")
plt.xlabel("Episodes"); plt.ylabel("Reward"); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print(f"Total Cliff Falls: Q-Learning={q_falls} | SARSA={s_falls}")

## 6. Development Tip: Using Modules
In a production setting, you should keep your environments and algorithms in separate `.py` files. This allows for testing and reuse across multiple notebooks.

In [ ]:
# Example of how you would import if the files were in your path:
# from environments import GridWorld, CliffWalking
# from algorithms import td0, run_agent